# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


# Answers

1.

In [ ]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-4, 4, 100)
def p_i(theta, a, b): return 1 / (1 + np.exp(-a * (theta - b)))

fig = go.Figure()
# a = 1.0, varying b
for b in [-1, 0, 1]:
    fig.add_trace(go.Scatter(x=theta, y=p_i(theta, 1.0, b), name=f"a=1.0, b={b}"))
# a = 2.0, varying b
for b in [-1, 0, 1]:
    fig.add_trace(go.Scatter(x=theta, y=p_i(theta, 2.0, b), name=f"a=2.0, b={b}", line=dict(dash='dash')))

fig.update_layout(title="2PL IRT Model Response Probabilities", xaxis_title="Theta", yaxis_title="P(Y=1 | Theta)")
fig.show()

2.

The likelihood of a single new response $y_k$ is a Bernoulli mass function,

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

The joint likelihood for the running history vector $\mathbf{y}^{(k)}$ is the product of independent contributions,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^k \left[ \frac{1}{1+e^{-a_j(\theta-b_j)}} \right]^{y_j} \left[ 1 - \frac{1}{1+e^{-a_j(\theta-b_j)}} \right]^{1 - y_j}$$

3.

Using Bayes' theorem, the recursive relationship drops the marginal likelihood denominator,

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

4.

If $y_k = 1$ for a large $b_k$, the likelihood $p_k(\theta)$ is near $0$ for low $\theta$ and approaches $1$ for high $\theta$. Multiplying the prior density by this monotonically increasing likelihood heavily penalizes the density at low $\theta$ values, mathematically pulling the peak of the posterior density to the right.

5.

The discrimination parameter $a_k$ dictates the steepness of the logistic likelihood curve.  

Very large $a_k$ - The likelihood resembles a step function, aggressively chopping off density below $b_k$. This drastically reduces posterior variance, creating a sharper density.

Very small $a_k$ - The likelihood is nearly flat. Multiplication by a flat function preserves the prior's shape, leaving the variance and sharpness largely unchanged.

6.

Numerical Implementation of a Running GridDefine a discrete, equally spaced array of $\theta$ values. Initialize the prior density array using the standard normal PDF evaluated at the grid points.  Upon observing $y_k$, evaluate the likelihood $L(y_k \mid \theta)$ at every grid point.Multiply the prior array by the likelihood array element wise to get the unnormalized posterior.

Sequential Normalization - Integrate the unnormalized array using the trapezoidal rule. Divide every element in the unnormalized array by this scalar integral to ensure the density sums to 1.

7.



In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

np.random.seed(42)
n = 20
theta_true = 0.75
theta_grid = np.linspace(-5, 5, 1000)
prior = norm.pdf(theta_grid, 0, 1)

a_vals = np.random.uniform(0.5, 2.0, n)
b_vals = np.random.normal(0, 1, n)

mean_estimates = []
map_estimates = []

for k in range(n):
    # Simulate response
    p_true = 1 / (1 + np.exp(-a_vals[k] * (theta_true - b_vals[k])))
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Likelihood and Update
    p_grid = 1 / (1 + np.exp(-a_vals[k] * (theta_grid - b_vals[k])))
    likelihood = (p_grid**y_k) * ((1 - p_grid)**(1 - y_k))
    unnormalized = prior * likelihood

    # Normalize
    prior = unnormalized / np.trapezoid(unnormalized, theta_grid)

    # Estimates
    mean_est = np.trapezoid(theta_grid * prior, theta_grid)
    map_est = theta_grid[np.argmax(prior)]

    mean_estimates.append(mean_est)
    map_estimates.append(map_est)

# Plotting
fig = go.Figure()
fig.add_trace(go.Scatter(y=mean_estimates, mode='lines+markers', name='Posterior Mean'))
fig.add_trace(go.Scatter(y=map_estimates, mode='lines+markers', name='MAP'))
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Theta")
fig.update_layout(title="Sequential Estimators over 20 Items", xaxis_title="Step k", yaxis_title="Estimate")
fig.show()

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

# Answers

1.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

x = np.linspace(0, 1, 500)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 1, 1), name="Uninformative (1,1)"))
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 2, 8), name="Right-skewed (2,8)"))
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 8, 2), name="Left-skewed (8,2)"))
fig.update_layout(title="Beta Distribution PDFs", xaxis_title="Theta", yaxis_title="Density")
fig.show()

2.

Sequential Likelihood and Joint History Single response likelihood,

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Joint likelihood for the history,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^k \theta^{y_j} (1 - \theta)^{1 - y_j} = \theta^{\sum_{j=1}^k y_j} (1 - \theta)^{k - \sum_{j=1}^k y_j}$$

3.

Closed-Form Analytical Updates (Conjugacy)Using Bayes' Theorem,

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot L(y_k \mid \theta)$$

$$\propto \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right] \cdot \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right]$$

$$= \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$
This matches the exact structural form of a Beta density. Therefore, the updates are closed-form arithmetic,

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

The Posterior Mean at step $k$ is:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

4.

Dynamic Shifting Mechanics
An observed click ($y_k=1$) adds 1 to $\alpha$, increasing the numerator of the expected value and mathematically shifting the peak to the right. A non-click ($y_k=0$) adds 1 to $\beta$, increasing the denominator and shifting the peak left. Unlike the 2PL IRT model which requires computational grid arrays and integrals, conjugate updates require $O(1)$ arithmetic addition with no numerical approximation.  

5.

Running Point EstimatorsRunning Posterior Mean,

$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$  

Running MAP,

$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$ (for $\alpha_k, \beta_k > 1$)

7.

In [ ]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
n = 100
theta_true = 0.35
alpha, beta = 1, 1

mean_ests, map_ests = [], []

for k in range(n):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha += y_k
    beta += (1 - y_k)

    mean_ests.append(alpha / (alpha + beta))
    # MAP bounded to avoid negative/invalid parameters in early stages
    map_val = (alpha - 1) / (alpha + beta - 2) if (alpha + beta) > 2 else alpha / (alpha + beta)
    map_ests.append(map_val)

fig = go.Figure()
fig.add_trace(go.Scatter(y=mean_ests, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(y=map_ests, mode='lines', name='MAP'))
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Theta")
fig.update_layout(title="Beta-Binomial Estimates over 100 Impressions", xaxis_title="Step k")
fig.show()

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

# Answers

1.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_grid = np.linspace(0.01, 1.0, 500)
prior_density = beta.pdf(theta_grid, 8, 1.5)

fig = go.Figure(data=go.Scatter(x=theta_grid, y=prior_density))
fig.update_layout(title="Initial Beta(8, 1.5) Prior", xaxis_title="Theta", yaxis_title="Density")
fig.show()

2.

Structural Likelihood Formulation

Given $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$, taking the natural log yields $\ln(y_k) = \ln(\theta K_{\text{nominal}}) + \epsilon_k$. Since $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, $\ln(y_k)$ is normally distributed.

The single measurement log-normal likelihood is,

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

The joint likelihood,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^k \frac{1}{y_j \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln y_j - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

3.

An exact analytical solution does not exist because the Beta distribution  is not a conjugate prior for the log-normal measurement likelihood (exponential terms involving $\ln(\theta)$). They cannot be algebraically merged into a recognized closed-form distribution family.

Recursive relationship,

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot \exp\left( - \frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

4.

Running Posterior Mean,

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

Running MAP,

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0,1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

5.

Define a dense numerical grid theta_grid = np.linspace(0.01, 1.0, 1000).

Exclude 0 strictly to avoid log(0) errors, matching boundary limits.

Evaluate the prior at these points.

For each new $y_k$, evaluate the log-normal likelihood vector across theta_grid.

Multiply the prior grid by the likelihood grid to form the unnormalized posterior.

Perform the sequential normalization step, compute Z = np.trapezoid(unnormalized_grid, theta_grid).

Divide the unnormalized_grid by Z.

6.



In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta, lognorm

np.random.seed(42)
n = 15
K_nom = 50.0
sigma = 0.15
theta_true = 0.68

theta_grid = np.linspace(0.01, 1.0, 1000)
posterior = beta.pdf(theta_grid, 8, 1.5)

mean_ests, map_ests = [], []
history_pdfs = {0: posterior.copy()}

for k in range(1, n + 1):
    # Simulate log-normal sensor reading
    y_k = theta_true * K_nom * np.exp(np.random.normal(0, sigma))

    # Likelihood and grid update
    likelihood = (1 / (y_k * sigma * np.sqrt(2 * np.pi))) * \
                 np.exp(-((np.log(y_k) - np.log(theta_grid * K_nom))**2) / (2 * sigma**2))
    unnormalized = posterior * likelihood
    posterior = unnormalized / np.trapezoid(unnormalized, theta_grid)

    mean_ests.append(np.trapezoid(theta_grid * posterior, theta_grid))
    map_ests.append(theta_grid[np.argmax(posterior)])

    if k in [1, 2, 5, 10, 15]:
        history_pdfs[k] = posterior.copy()

# Plot 1: Curves
fig1 = go.Figure()
for k, pdf in history_pdfs.items():
    fig1.add_trace(go.Scatter(x=theta_grid, y=pdf, name=f"Step {k}"))
fig1.update_layout(title="Posterior Density Evolution", xaxis_title="Theta", yaxis_title="Density")
fig1.show()

# Plot 2: Convergence Tracking
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(range(1, 16)), y=mean_ests, name='Mean'))
fig2.add_trace(go.Scatter(x=list(range(1, 16)), y=map_ests, name='MAP'))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red")
fig2.update_layout(title="Estimator Convergence", xaxis_title="Step k", yaxis_title="Theta Estimate")
fig2.show()

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

# Answers

1.

By the Law of Total Probability,

$$p(x_i) = \sum_{k=1}^K P(X_i = x_i \mid C_i = k) P(C_i = k)$$

Substituting the known distributions,

$$p(x_i) = \sum_{k=1}^K \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \phi_k$$

This density is called a Gaussian mixture density because it is formulated as a linear combination of $K$ distinct Gaussian component probability density functions, weighted by their respective prior mixing proportions $\phi_k$.

2.

Applying Bayes' rule,

$$P(C_i=k \mid X_i=x_i) = \frac{P(X_i=x_i \mid C_i=k) P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i \mid C_i=j) P(C_i=j)}$$

Substituting the multivariate Gaussian density and prior probabilities,

$$\gamma_{ik} = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$

$\gamma_{ik}$ is interpreted as a posterior probability because it represents the updated belief that point $x_i$ belongs to cluster $k$ after explicitly observing the spatial evidence of $x_i$ relative to the clusters' means and variances.  

3.

The conditional expectation of an indicator variable is equal to the conditional probability of the event,

$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = 1 \cdot P(C_i=k \mid X_i=x_i) + 0 \cdot P(C_i \neq k \mid X_i=x_i)$$

$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = \gamma_{ik}$$

Vectorizing this across all $K$ clusters yields,

$$\mathbb{E}[Z_i \mid X_i=x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

Conclusively, soft clustering assigns a vector of probabilistic expectations rather than a singular deterministic identity.  

4.

Soft clustering outputs a probability distribution vector $(\gamma_{i1}, \dots, \gamma_{iK})$ quantifying the fractional belief of membership across all clusters. Hard clustering forces a mutually exclusive assignment by condensing the soft vector into a definitive label $\widehat{C}_i$, selecting the cluster that achieved the absolute maximum posterior responsibility.  

5.

By the fundamental definition of the normal distribution, the expected value of a random variable drawn from $\mathscr{N}(\mu_k, \Sigma_k)$ is its mean parameter,

$$\mathbb{E}[X_i \mid C_i=k] = \int x_i \mathscr{N}(x_i \mid \mu_k, \Sigma_k) dx_i = \mu_k$$

$\mu_k$ acts as the geometric center of cluster $k$ because it is the expected spatial coordinate for data points generated by that specific multivariate Gaussian process.

Comparison - $\mathbb{E}[Z_i \mid X_i=x_i]$ operates in the categorical label space representing cluster membership probabilities given a fixed data coordinate.

Conversely, $\mathbb{E}[X_i \mid C_i=k]$ operates in the spatial coordinate space representing the geometric mean location given a fixed categorical cluster identity.  

6.

Taking the natural logarithm of the joint product,

$$\ln p(X, Z) = \ln \left( \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K \ln \left( \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \ln \phi_k + \ln \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

If $z_{ik}$ were known, this expression would be easy to maximize because the log-likelihood decoupling ensures the parameters for cluster $k$ ($\phi_k, \mu_k, \Sigma_k$) can be optimized entirely independently of the parameters for any other cluster.  

7.

Taking the expectation of $\ell_c$ with respect to the posterior of the latent variables,

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \ln \phi_k + \ln \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

The E-step acts as a conditional update because it evaluates the posterior probability $\gamma_{ik}$ using the current iteration's parameter estimates, dynamically shifting the cluster membership weights based on the newly calculated spatial compatibilities.  

8.

The maximization of $Q$ yields the standard updates. The responsibility $\gamma_{ik}$ acts as a fractional weight in these formulas because, rather than entirely allocating $x_i$ to a single cluster, the algorithm allocates exactly $\gamma_{ik} \times 100\%$ of point $x_i$ to update cluster $k$'s specific mean and covariance structures.  

9.

Gaussian Mixture Modeling clustering can be fundamentally viewed as an iterative process of conditional updating. Initially, the mixture weight $\phi_k$ defines the prior probability of belonging to cluster $k$ before any spatial data is observed. The Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ then serves as a likelihood function measuring how physically compatible $x_i$ is with cluster $k$'s current geometry. Using Bayes' rule, the responsibility $\gamma_{ik}$ fuses these two metrics into the posterior probability of cluster $k$ after observing the evidence $x_i$. Grouping these responsibilities creates the soft assignment vector $\mathbb{E}[Z_i \mid X_i=x_i]$. Finally, the M-step structurally updates the physical cluster parameters using these posterior membership probabilities as fractional weights. Thus, GMM is intrinsically probabilistic clustering defined by the continuous conditional expectations of latent cluster membership variables.

10.



In [ ]:
import numpy as np
import pandas as pd
import io
import plotly.graph_objects as go
import plotly.express as px
from google.colab import files
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.n_components = n_components
        self.gmm = GaussianMixture(n_components=self.n_components, covariance_type='full', random_state=42)
        self.scaler = StandardScaler()

    def fit_evaluate(self, df):
        # Data Splitting and Scaling
        X = self.scaler.fit_transform(df[['PURCHASES', 'CREDIT_LIMIT']])
        X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

        # EM Execution
        self.gmm.fit(X_train)
        print(f"Converged: {self.gmm.converged_}")
        print(f"Iterations: {self.gmm.n_iter_}")

        # Out-of-Sample Performance
        test_score = self.gmm.score(X_test)
        print(f"Average Log-Likelihood on Test Set: {test_score:.4f}")

        self._visualize(X_train, X_test)

    def _visualize(self, X_train, X_test):
        # 1. 2D Density Heatmap
        fig1 = px.density_heatmap(
            x=X_train[:, 0], y=X_train[:, 1],
            marginal_x="histogram", marginal_y="histogram",
            title="Empirical 2D Density Heatmap (Training)",
            labels={'x': 'Scaled PURCHASES', 'y': 'Scaled CREDIT_LIMIT'}
        )
        fig1.show()

        # Grid creation for contours
        x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
        y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid = np.c_[xx.ravel(), yy.ravel()]

        # Compute responsibilities for the background contour
        Z = self.gmm.predict(grid)
        Z = Z.reshape(xx.shape)

        # 2. Training Assignment Plot
        fig2 = go.Figure()
        fig2.add_trace(go.Contour(
            x=np.linspace(x_min, x_max, 200), y=np.linspace(y_min, y_max, 200), z=Z,
            colorscale='Viridis', opacity=0.3, showscale=False, hoverinfo='skip'
        ))
        fig2.add_trace(go.Scatter(
            x=X_train[:, 0], y=X_train[:, 1], mode='markers',
            marker=dict(color=self.gmm.predict(X_train), size=5, colorscale='Viridis', line=dict(width=0.5, color='white')),
            name='Train Data'
        ))
        fig2.update_layout(title="Training Assignment Plot", xaxis_title="Scaled PURCHASES", yaxis_title="Scaled CREDIT_LIMIT")
        fig2.show()

        # 3. Test Assignment Plot
        fig3 = go.Figure()
        fig3.add_trace(go.Contour(
            x=np.linspace(x_min, x_max, 200), y=np.linspace(y_min, y_max, 200), z=Z,
            colorscale='Viridis', opacity=0.3, showscale=False, hoverinfo='skip'
        ))
        fig3.add_trace(go.Scatter(
            x=X_test[:, 0], y=X_test[:, 1], mode='markers',
            marker=dict(color=self.gmm.predict(X_test), size=6, colorscale='Viridis', symbol='cross'),
            name='Test Data'
        ))
        fig3.update_layout(title="Test Assignment Plot (Out-of-Sample)", xaxis_title="Scaled PURCHASES", yaxis_title="Scaled CREDIT_LIMIT")
        fig3.show()

# --- COLAB FILE UPLOAD EXECUTION ---
print("Please upload the Kaggle CSV file you extracted:")
uploaded = files.upload()

# Get the exact filename of whatever you just uploaded
filename = list(uploaded.keys())[0]

# Read the uploaded file into pandas
dataset = pd.read_csv(io.BytesIO(uploaded[filename])).dropna(subset=['PURCHASES', 'CREDIT_LIMIT'])
print(f"\nSuccessfully loaded {filename}! Running GMM Segmenter...\n")

# Execute the GMM fitting and plotting
segmenter = GMMFinancialSegmenter(n_components=3)
segmenter.fit_evaluate(dataset)

Please upload the Kaggle CSV file you extracted:


Saving CC GENERAL.csv to CC GENERAL.csv

Successfully loaded CC GENERAL.csv! Running GMM Segmenter...

Converged: True
Iterations: 19
Average Log-Likelihood on Test Set: -1.6465
